In [1]:
import torch
import torch_xla.core.xla_model as xm
import torch_xla
import warnings

# Suppress non-critical warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Aliasing torch_xla to torch.xla
torch.xla = torch_xla

device = xm.xla_device()
print(f"Using device: {device}")

Using device: xla:0


In [2]:
from datasets import load_dataset
import os

dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
os.makedirs("data", exist_ok=True)
dataset.to_json("data/dolly_train.json", orient="records", lines=True)

Creating json from Arrow format:   0%|          | 0/16 [00:00<?, ?ba/s]

12982756

In [3]:
print(type(dataset))

<class 'datasets.arrow_dataset.Dataset'>


In [4]:
import pandas as pd
sampledata=pd.DataFrame(dataset)
sampledata

,instruction,context,response,category
0,When did Virgin Australia start operating?,"Virgin Australia, the trading name of Virgin A...",Virgin Australia commenced services on 31 Augu...,closed_qa
1,Which is a species of fish? Tope or Rope,,Tope,classification
2,Why can camels survive for long without water?,,Camels use the fat in their humps to keep them...,open_qa
3,"Alice's parents have three daughters: Amy, Jes...",,The name of the third daughter is Alice,open_qa
4,When was Tomoaki Komorida born?,Komorida was born in Kumamoto Prefecture on Ju...,"Tomoaki Komorida was born on July 10,1981.",closed_qa
...,...,...,...,...
15006,How do i accept the change,,Embrace the change and see the difference,brainstorming
15007,What is a laser and who created it?,A laser is a device that emits light through a...,A laser is a device that emits light from an e...,summarization
15008,What is the difference between a road bike and...,,Road bikes are built to be ridden on asphalt a...,open_qa
15009,How does GIS help in the real estate investmen...,,"Real estate investors depend on precise, accur...",general_qa


In [5]:
# 1. Load Pretrained model & tokenizer
import torch
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [6]:
# Converting raw text into trainng samples
def format_and_tokenize(example):

    text = f"""### Instruction:{example['instruction']}
               ### Context:{example.get('context','')}
               ### Response: {example['response']}"""

    return tokenizer(
        #Converts text into token IDs,
        #Ensures sequences fit within the model’s context window
        text,
        truncation=True,
        max_length=512,   # THIS is where we control memory
        padding=False,
    )

dataset = load_dataset("json", data_files="data/dolly_train.json")["train"]
dataset = dataset.shuffle(seed=42).select(range(1000))  # reduce steps + memory
dataset = dataset.map(
    format_and_tokenize,
    remove_columns=dataset.column_names,
)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [7]:
# Injecting LoRA into the Transformer (Core Step)
from peft import LoraConfig, get_peft_model

# LoRA config
lora_config = LoraConfig(
    r=8,# rank controls — How much the model can deviate from its original behavior

    lora_alpha=16,
    # Scaling the update - LoRA adds a small extra update to the existing weight
    # lora_alpha decides how big that update is.
    # alpha too small- update barely changes base model and viceversa

    target_modules=["q_proj", "v_proj"],
    #decide which parts of the Transformer are allowed to learn during fine-tuning

    lora_dropout=0.05, # Regularizing the adoption

    bias="none", # bias="none" keeps all bias terms frozen, only Lora matrices are trained

    task_type="CAUSAL_LM", #  saying focus on next word prediction task
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


### Prep the model for training

In [8]:

model.enable_input_require_grads()# input embeddings don’t track gradients (they are  along with weights)
#This line ensures that even though the input embeddings aren't directly trained, gradients can still flow through them to the trainable LoRA parameters.

model.gradient_checkpointing_enable() #Saves memory by not storing everything; it recalculates some values when needed.
#Memoization trades memory for time (store results to save computation time).
#Gradient checkpointing trades time for memory (re-compute activations to save memory).

model.config.use_cache = False # Turns off caching so gradients are calculated correctly while training.

### Defining the Supervised Fine-Tuning Setup

In [9]:
from transformers import TrainingArguments

# Training arguments
training_args = TrainingArguments(
    output_dir="./outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=False, # Changed to False for CPU compatibility
    logging_steps=5,
)

###Running LoRA + SFT Training

In [10]:
from transformers import TrainingArguments

# Training arguments
training_args = TrainingArguments(
    output_dir="./outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=False, # Changed to False to resolve fp16 compatibility issue with XLA/accelerate
    logging_steps=5,
)

In [ ]:
from trl import SFTTrainer

# Ensure the model is moved to the XLA device before training
model.to(device)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)
trainer.train()

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


In [ ]:
# Saving the LoRA Adapter
trainer.model.save_pretrained("lora-adapter")
tokenizer.save_pretrained("lora-adapter")

### Merging LoRA with the Base Model+ inferenceing

In [ ]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0", dtype="auto", device_map="auto")
model = PeftModel.from_pretrained(base_model, "lora-adapter")
# Save merged model
model.save_pretrained("merged_model_new")

In [ ]:
model = AutoModelForCausalLM.from_pretrained("merged_model_new")
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

inputs = tokenizer(
    "Explain LoRA fine-tuning in simple terms for a product manager.",
    return_tensors="pt"
).to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)